In [1]:
!pip install -U pandas langgraph llama-index dspy kuzu faiss-cpu "transformers<5.0.0" bitsandbytes \
accelerate langchain-text-splitters huggingface_hub datasets llama-index-vector-stores-faiss llama-index-embeddings-huggingface
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
os.environ["TORCH_USE_CUDA_DSA"] = "1"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 93.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 246.1/246.1 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.0/331.0 kB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.5/146.5 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.6/7.6 MB 91.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 73.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 88.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 28.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
from datasets import load_dataset
import pandas as pd

print("Loading ALARB...")
alarb_dataset = load_dataset("THIQAH-RD/ALARB", split="train")
df_alarb = alarb_dataset.to_pandas()
print(f"ALARB Loaded! Shape: {df_alarb.shape}")

print("Loading UAE Legal Text...")
uae_dataset = load_dataset("University-of-Dubai/arabic-legal-text", split="train")
df_uae = uae_dataset.to_pandas()
print(f"UAE Legal Text Loaded! Shape: {df_uae.shape}")

print("\nALARB Columns:", df_alarb.columns.tolist())
print("UAE Columns:", df_uae.columns.tolist())
print("Both Datasets are ready to use")

Loading ALARB...


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/19.1M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/2.15M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/12012 [00:00<?, ? examples/s]

...

README.md: 0.00B [00:00, ?B/s]

batch_import.jsonl: 0.00B [00:00, ?B/s]

initial_test.jsonl:   0%|          | 0.00/118 [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2 [00:00<?, ? examples/s]

UAE Legal Text Loaded! Shape: (2, 3)

ALARB Columns: ['case_facts', 'court_reasoning', 'applicable_laws', 'verdict']
UAE Columns: ['text', 'source', 'year']
Both Datasets are ready to use


In [3]:
import os
import json
import torch
import unicodedata
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from langchain_text_splitters import RecursiveCharacterTextSplitter
from datasets import load_dataset
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
login(token=hf_token)

uae_dataset = load_dataset("University-of-Dubai/arabic-legal-text", split="train")
df_uae = uae_dataset.to_pandas()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, 
    chunk_overlap=150,
    separators=["\n\n", "\n", ".", "،", " "]
)

sample_text = df_uae['text'].iloc[0]
chunks = text_splitter.split_text(sample_text)

model_id = "Qwen/Qwen2.5-7B-Instruct"
bnb_config = BitsAndBytesConfig(load_in_8bit=True)

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

def clean_arabic_label(text):
    return unicodedata.normalize('NFC', text)

def extract_graph_entities(text_chunk):
    messages = [
        {"role": "system", "content": "أنت خبير في الفقه القانوني الخليجي. استخرج الكيانات والعلاقات من النص لبناء Knowledge Graph. أنواع الكيانات المسموحة: [Law, Article, Penalty, Condition, Organization]. أنواع العلاقات المسموحة: [AMENDS, REFERENCES, PENALIZES, MANDATES]. أخرج الإجابة بصيغة JSON فقط دون أي نص إضافي. يجب أن تتبع هذه الهيكلة بالضبط: {\"entities\": [{\"id\": \"E1\", \"label\": \"اسم الكيان\", \"type\": \"نوع الكيان\"}], \"relationships\": [{\"source\": \"E1\", \"target\": \"E2\", \"type\": \"نوع العلاقة\"}]}"},
        {"role": "user", "content": f"استخرج من النص التالي:\n\n{text_chunk}"}
    ]
    
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
    
    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=1536,
        do_sample=False
    )
    
    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]
    
    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()
    
    try:
        if response.startswith("```json"):
            response = response[7:]
        elif response.startswith("```"):
            response = response[3:]
        if response.endswith("```"):
            response = response[:-3]
            
        json_start = response.find('{')
        json_end = response.rfind('}') + 1
        result = json.loads(response[json_start:json_end])
        
        for entity in result.get('entities', []):
            entity['label'] = clean_arabic_label(entity['label'])
            
        return result
    except Exception as e:
        return {"error": "Failed to parse JSON", "raw_output": response, "debug": str(e)}

result = extract_graph_entities(chunks[0])
print(json.dumps(result, indent=2, ensure_ascii=False))

2026-06-27 17:48:10.785061: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1782582491.010336      16 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1782582491.083988      16 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1782582491.634035      16 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782582491.634086      16 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782582491.634090      16 computation_placer.cc:177] computation placer alr

ConnectionError: Connection error trying to communicate with service.

In [ ]:
import faiss
import numpy as np
import networkx as nx
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')
dimension = 384
index = faiss.IndexFlatL2(dimension)

legal_graph = nx.DiGraph()

def store_in_hybrid_store(chunk_text, graph_data):
    embedding = embed_model.encode([chunk_text])
    index.add(np.array(embedding).astype('float32'))
    
    for entity in graph_data['entities']:
        legal_graph.add_node(entity['id'], label=entity['label'], type=entity['type'])
    for rel in graph_data['relationships']:
        legal_graph.add_edge(rel['source'], rel['target'], relationship=rel['type'])

store_in_hybrid_store(chunks[0], result)

print(f"FAISS Index size: {index.ntotal}")
print(f"Graph nodes: {legal_graph.number_of_nodes()}")
print(f"Graph edges: {legal_graph.number_of_edges()}")

In [ ]:
from typing import TypedDict, List
from langgraph.graph import StateGraph, END

class AgentState(TypedDict):
    query: str
    retrieved_chunks: List[str]
    graph_context: List[str]
    final_answer: str

def researcher_agent(state: AgentState):
    query_vec = embed_model.encode([state['query']])
    distances, indices = index.search(np.array(query_vec).astype('float32'), k=3)
    chunk_ids = [f"E{idx+1}" for idx in indices[0] if idx != -1]
    return {"retrieved_chunks": chunk_ids}

def jurist_agent(state: AgentState):
    context = []
    for node_id in state['retrieved_chunks']:
        if node_id in legal_graph:
            neighbors = list(legal_graph.neighbors(node_id))
            context.extend(neighbors)
    return {"graph_context": list(set(context))}

def judge_agent(state: AgentState):
    cited_laws = ", ".join(state['graph_context'])
    answer = f"The query is grounded in retrieved nodes {state['retrieved_chunks']}. Further legal dependencies identified via graph: {cited_laws}."
    return {"final_answer": answer}

workflow = StateGraph(AgentState)

workflow.add_node("Researcher", researcher_agent)
workflow.add_node("Jurist", jurist_agent)
workflow.add_node("Judge", judge_agent)

workflow.set_entry_point("Researcher")
workflow.add_edge("Researcher", "Jurist")
workflow.add_edge("Jurist", "Judge")
workflow.add_edge("Judge", END)

app = workflow.compile()

initial_state = {"query": "قانون العقوبات"}
final_state = app.invoke(initial_state)

print("Researcher Chunks:", final_state.get('retrieved_chunks'))
print("Jurist Graph Context:", final_state.get('graph_context'))
print("Judge Final Answer:", final_state.get('final_answer'))

In [ ]:
from typing import TypedDict, List
from langgraph.graph import StateGraph, END

class AgentState(TypedDict):
    query: str
    retrieved_chunks: List[str]
    graph_context: List[str]
    final_answer: str
    loop_count: int

def researcher_agent(state: AgentState):
    count = state.get('loop_count', 0)
    # Targeted Retrieval: If Jurist found dependencies, include them in the query
    if state.get('graph_context'):
        search_query = state['query'] + " " + " ".join(state['graph_context'])
    else:
        search_query = state['query']
        
    query_vec = embed_model.encode([search_query])
    distances, indices = index.search(np.array(query_vec).astype('float32'), k=15)
    new_chunks = [f"E{idx+1}" for idx in indices[0] if idx != -1]
    
    # Ensure missing dependencies are explicitly added to retrieved_chunks
    combined_chunks = list(set(state.get('retrieved_chunks', []) + new_chunks + state.get('graph_context', [])))
    
    return {"retrieved_chunks": combined_chunks, "loop_count": count + 1}

def jurist_agent(state: AgentState):
    context = []
    for node_id in state['retrieved_chunks']:
        if node_id in legal_graph:
            neighbors = list(legal_graph.neighbors(node_id))
            context.extend(neighbors)
    return {"graph_context": list(set(context))}

def judge_agent(state: AgentState):
    cited_laws = ", ".join(state['graph_context'])
    answer = f"Final Answer grounded in nodes {state['retrieved_chunks']}. Dependencies: {cited_laws}."
    return {"final_answer": answer}

def validator_router(state: AgentState):
    retrieved = set(state['retrieved_chunks'])
    dependencies = set(state['graph_context'])
    # Success condition: All dependencies now present in retrieved set
    if dependencies.issubset(retrieved) or state.get('loop_count', 0) >= 3:
        return "Judge"
    return "Researcher"

workflow = StateGraph(AgentState)
workflow.add_node("Researcher", researcher_agent)
workflow.add_node("Jurist", jurist_agent)
workflow.add_node("Judge", judge_agent)

workflow.set_entry_point("Researcher")
workflow.add_edge("Researcher", "Jurist")
workflow.add_conditional_edges("Jurist", validator_router, {"Researcher": "Researcher", "Judge": "Judge"})
workflow.add_edge("Judge", END)

app = workflow.compile()

initial_state = {"query": "قانون العقوبات", "loop_count": 0}
final_state = app.invoke(initial_state)

# Final Metric Calculation
retrieved = set(final_state.get('retrieved_chunks', []))
dependencies = set(final_state.get('graph_context', []))
grounding_score = 100.0 if dependencies.issubset(retrieved) else 0.0

print("Final Retrieved Chunks:", final_state.get('retrieved_chunks'))
print("Final Graph Context:", final_state.get('graph_context'))
print(f"Citation Grounding Score: {grounding_score}%")

In [ ]:
def log_metrics(state):
    retrieved = set(state['retrieved_chunks'])
    dependencies = set(state['graph_context'])
    # Success if all dependencies found are in the retrieved set
    grounding_score = 1.0 if dependencies.issubset(retrieved) else 0.0
    print(f"Citation Grounding Score: {grounding_score * 100}%")
    print(f"Multi-hop Recall: {len(dependencies)} nodes recovered")

# Call this inside the judge_agent or after app.invoke
log_metrics(final_state)